# Django Form

- - -
## Django Form
- - -
#### HTML 'form'
지금까지 사용자로부터 데이터를 제출 받기위해 활용한 방법<br>
그러나 비정상적 혹은 악의적인 요청을 필터링 할 수 없음

<br>

#### 유효성 검사
수집한 데이터가 정확하고 유효한지 확인하는 과정<br>
숫자면 숫자, 문자면 문자

### Form class

#### Django Form
사용자 입력 데이터를 수집하고,<br>
처리 및 유효성 검사를 수행하기 위한 도구

In [ ]:
# articles/forms.py

from django import forms

class ArticleForm(forms.Form):
    title = forms.CharField(max_length=10)
    content = forms. CharField()

In [ ]:
# articles/views.py

from .forms import ArticleForm

def new(request):
    form = ArticleForm()
    context = {
        'form': form,
    }

    return render(request, 'articles/new.html', context)

In [ ]:
<!-- articles/new.html -->

<h1>NEW</h1>
<form action="{% url 'articles: create' %}" method="POST">
  {% csrf token %}
  {{ form }}
  <input type="submit">
</form>

### Widgets

HTML 'input' element의 표현을 담당<br>
단순히 input요소의 속성 및 출력되는 부분을 변경하는 것

- - -
## Django Model Form
- - -
<br>

| Form | ModelForm |
| :---: | :---: |
| 사용자 입력 데이터를<br> DB에 저장하지 않을 때<br> ex) 검색, 로그인 | 사용자 입력 데이터를<br>DB에 저장해야 할 때<br> ex) 게시글 작성, 회원가입 |

<br>

### ModelForm
Model과 Form을 자동으로 생성해주는 기능을 제공

In [ ]:
# articles/forms.py

from django import forms
from .models import Article

class ArticleForm(forms.ModelForm):
    class Meta:
        model = Article
        fields = '__all__'

### Meta class
ModelForm의 정보를 작성하는 곳

#### 'fields' 및 'exclude'속성
fields로 속성을 지정 혹은 <br>
exclude로 포함하지 않도록 지정할 수도 있음

In [ ]:
# articles/forms.py

class ArticleForm(forms.ModelForm):
    class Meta:
        model = Article
        exclude = ('title',)
        # fields = ('title',)

### ModelForm 적용

#### ModelForm을 적용한 create 로직

In [ ]:
# articles/views.py

from .forms import ArticleForm

def create(request):
    form = ArticleForm(request.POST)
    if form.is_valid():
        article = form. save()
        return redirect('articles:detail', article.pk)
    context = {
        'form': form,
    }
    return render(request, 'articles/new.html', context)

#### is_valid()
여러 유효성 검사를 실행하고, 데이터가 유효한지 여부를 Boolean으로 반환

#### ModelForm을 적용한 edit 로직

In [ ]:
# articles/views.py

def edit(request, pk):
    article = Article.objects.get(pk=pk)
    form = ArticleForm(instance=article)
    context = {
        'article': article,
        'form': form,
    }

    return render(request, 'articles/edit.html', context)

In [ ]:
<!-- articles/edit.html -->
<h1>EDIT</h1>
<form action="{% url 'articles:update' article.pk %}" method="POST">
  {% csrf_token %}
  {{ form }}
  <input type="submit">
</form>

In [ ]:
# articles/views.py

def update(request, pk):
    article = Article.objects.get(pk=pk)
    form = ArticleForm(request.POST, instance=article)
    if form.is_valid():
        form.save()
        return redirect('articles:detail', article.pk)
    context = {
        'article': article,
        'form': form,
    }
    return render(request, 'articles/edit.html', context)

### save 메서드
데이터베이스 객체를 만들고 저장하는 ModelForm의 인스턴스 메서드

- - -
## HTTP 요청 다루기
- - -

###  View 함수 구조 변화

### new & create 함수 결합

| 공통점 | 차이점 |
| :---: | :---: |
| "데이터 생성을 구현하기 위함" | "new는 GET method 요청만을,<br> create는 POST method 요청만을 처리" |

In [ ]:
def new(request):
    form = ArticleForm()
    context = {
        'form': form,
    }
    return render(request, 'articles/new.html', context)

In [ ]:
def create(request):
    form = ArticleForm(request.POST)
    if form.is_valid():
        article = form.save()
        return redirect('articles:detail', article.pk)
    context = {
        'form': form,
    }
    return render(request, 'articles/new.html', context)

In [ ]:
def create(request):
    if request.method == 'POST':
        form = ArticleForm(request.POST)
        if form.is_valid():
            article = form.save()
            return redirect('articles:detail', article.pk)
    
    else:
        form = ArticleForm()
    context = {
        'form': form,
    }
    return render(request, 'articles/new.html', context)

### edit & update 함수 결합

In [ ]:
# articles/views.py

def update(request, pk):
    article = Article.objects.get(pk=pk)
    if request.method == 'POST':
        form = ArticleForm(request.POST, instance=article)
        if form.is_valid():
            form. save()
            return redirect('articles:detail', article.pk)

    else:
        form = ArticleForm(instance=article)
    context = {
        'article': article,
        'form': form,
    }
    return render(request, 'articles/update.html', context)